<a href="https://colab.research.google.com/github/Oct-o-Dev/DL/blob/main/LSTM_next_word_predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
faq =  """Which databases to use?
You can choose between a traditional relational database and a non-relational database. Let
us examine their differences.
Relational databases are also called a relational database management system (RDBMS) or
SQL database. The most popular ones are MySQL, Oracle database, PostgreSQL, etc.
Relational databases represent and store data in tables and rows. You can perform join
operations using SQL across different database tables.
Non-Relational databases are also called NoSQL databases. Popular ones are CouchDB,
Neo4j, Cassandra, HBase, Amazon DynamoDB, etc. [2]. These databases are grouped into
four categories: key-value stores, graph stores, column stores, and document stores. Join
operations are generally not supported in non-relational databases.
For most developers, relational databases are the best option because they have been around
for over 40 years and historically, they have worked well. However, if relational databases
are not suitable for your specific use cases, it is critical to explore beyond relational
databases. Non-relational databases might be the right choice if:
• Your application requires super-low latency.
• Your data are unstructured, or you do not have any relational data.
• You only need to serialize and deserialize data (JSON, XML, YAML, etc.).
• You need to store a massive amount of data.
Vertical scaling vs horizontal scaling
Vertical scaling, referred to as “scale up”, means the process of adding more power (CPU,
RAM, etc.) to your servers. Horizontal scaling, referred to as “scale-out”, allows you to scale
by adding more servers into your pool of resources.
When traffic is low, vertical scaling is a great option, and the simplicity of vertical scaling is
its main advantage. Unfortunately, it comes with serious limitations.
• Vertical scaling has a hard limit. It is impossible to add unlimited CPU and memory to a
single server.
• Vertical scaling does not have failover and redundancy. If one server goes down, the
website/app goes down with it completely.
Horizontal scaling is more desirable for large scale applications due to the limitations of
vertical scaling.
In the previous design, users are connected to the web server directly. Users will unable to
access the website if the web server is offline. In another scenario, if many users access the
web server simultaneously and it reaches the web server’s load limit, users generally
experience slower response or fail to connect to the server. A load balancer is the best
technique to address these problems."""

In [2]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer


In [3]:
tokenizer = Tokenizer()

In [5]:
tokenizer.fit_on_texts([faq])

In [24]:
len(tokenizer.word_index)

200

In [12]:
input_sequence = []
for sentence in faq.split('\n'):
  tokenized_sentence = tokenizer.texts_to_sequences([sentence])[0]

  for i in range(1,len(tokenized_sentence)):
    input_sequence.append(tokenized_sentence[:i+1])

In [13]:
input_sequence

[[67, 4],
 [67, 4, 1],
 [67, 4, 1, 32],
 [11, 33],
 [11, 33, 68],
 [11, 33, 68, 69],
 [11, 33, 68, 69, 8],
 [11, 33, 68, 69, 8, 70],
 [11, 33, 68, 69, 8, 70, 3],
 [11, 33, 68, 69, 8, 70, 3, 12],
 [11, 33, 68, 69, 8, 70, 3, 12, 5],
 [11, 33, 68, 69, 8, 70, 3, 12, 5, 8],
 [11, 33, 68, 69, 8, 70, 3, 12, 5, 8, 20],
 [11, 33, 68, 69, 8, 70, 3, 12, 5, 8, 20, 3],
 [11, 33, 68, 69, 8, 70, 3, 12, 5, 8, 20, 3, 12],
 [11, 33, 68, 69, 8, 70, 3, 12, 5, 8, 20, 3, 12, 71],
 [72, 73],
 [72, 73, 74],
 [72, 73, 74, 75],
 [3, 4],
 [3, 4, 6],
 [3, 4, 6, 34],
 [3, 4, 6, 34, 35],
 [3, 4, 6, 34, 35, 8],
 [3, 4, 6, 34, 35, 8, 3],
 [3, 4, 6, 34, 35, 8, 3, 12],
 [3, 4, 6, 34, 35, 8, 3, 12, 76],
 [3, 4, 6, 34, 35, 8, 3, 12, 76, 77],
 [3, 4, 6, 34, 35, 8, 3, 12, 76, 77, 78],
 [3, 4, 6, 34, 35, 8, 3, 12, 76, 77, 78, 29],
 [36, 12],
 [36, 12, 2],
 [36, 12, 2, 37],
 [36, 12, 2, 37, 38],
 [36, 12, 2, 37, 38, 39],
 [36, 12, 2, 37, 38, 39, 6],
 [36, 12, 2, 37, 38, 39, 6, 79],
 [36, 12, 2, 37, 38, 39, 6, 79, 80],
 [36, 

In [31]:
max_len = max([len(x) for x in input_sequence])

In [32]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
padded_input_sequences = pad_sequences(input_sequence , maxlen = max_len , padding='pre')

In [19]:
padded_input_sequences

array([[  0,   0,   0, ...,   0,  67,   4],
       [  0,   0,   0, ...,  67,   4,   1],
       [  0,   0,   0, ...,   4,   1,  32],
       ...,
       [  0,   0,   0, ..., 198,   1, 199],
       [  0,   0,   0, ...,   1, 199,  44],
       [  0,   0,   0, ..., 199,  44, 200]], dtype=int32)

In [20]:
X = padded_input_sequences[:,:-1]

In [21]:
X

array([[  0,   0,   0, ...,   0,   0,  67],
       [  0,   0,   0, ...,   0,  67,   4],
       [  0,   0,   0, ...,  67,   4,   1],
       ...,
       [  0,   0,   0, ...,   0, 198,   1],
       [  0,   0,   0, ..., 198,   1, 199],
       [  0,   0,   0, ...,   1, 199,  44]], dtype=int32)

In [22]:
y = padded_input_sequences[:,-1]

In [23]:
y

array([  4,   1,  32,  33,  68,  69,   8,  70,   3,  12,   5,   8,  20,
         3,  12,  71,  73,  74,  75,   4,   6,  34,  35,   8,   3,  12,
        76,  77,  78,  29,  12,   2,  37,  38,  39,   6,  79,  80,  12,
        81,  21,   4,  82,   5,  40,  15,  22,  41,   5,  83,  11,  33,
        84,  42,  85,  36,  86,  87,  12,  41,   3,   4,   6,  34,  35,
        88,   4,  38,  39,   6,  89,  91,  92,  93,  94,  21,  95,  44,
         4,   6,  96,  45,  98,  99, 100,  23, 101,  23, 102,  23,   5,
       103,  23,  42,   6,  46,  24, 104,  22,  20,   3,   4,  37, 105,
         3,   4,   6,   2,  47,  48, 106,  49,  26, 107, 108, 109, 110,
       111,   5, 112,  49,  26, 113, 114, 115,  16,   3,   4,  24, 116,
        25,  17, 117,  32, 118,  18,   9, 119,   1, 120, 121,   3,  20,
         3,   4, 122, 123,   2, 124, 125,  16,  17, 126, 127, 128,  50,
       129,  17,  15,   6, 130,  29,  11, 131,  24,  26, 132,   3,  15,
        11, 133,  51,   1, 134,   5, 135,  15, 136, 137, 138,  2

In [25]:
from tensorflow.keras.utils import to_categorical
y = to_categorical(y , num_classes = 201)

In [26]:
y.shape

(369, 201)

In [27]:
y[0]

array([0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

In [28]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding , LSTM , Dense

In [33]:
model = Sequential()
model.add(Embedding(201 , 100 , input_length=18))
model.add(LSTM(150))
model.add(Dense(201 , activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [34]:
model.compile(loss='categorical_crossentropy' , optimizer='adam' , metrics=['accuracy'])

In [36]:
model.build(input_shape=(None, max_len))
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 18, 100)        │        20,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 150)            │       150,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 201)            │        30,351 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 201,051 (785.36 KB)

 Trainable params: 201,051 (785.36 KB)

 Non-trainable params: 0 (0.00 B)

In [38]:
X.shape

(369, 17)

In [39]:
 X

array([[  0,   0,   0, ...,   0,   0,  67],
       [  0,   0,   0, ...,   0,  67,   4],
       [  0,   0,   0, ...,  67,   4,   1],
       ...,
       [  0,   0,   0, ...,   0, 198,   1],
       [  0,   0,   0, ..., 198,   1, 199],
       [  0,   0,   0, ...,   1, 199,  44]], dtype=int32)

In [40]:
len(X[0])

17

In [42]:
model.fit(X,y,epochs=100)

Epoch 1/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.0786 - loss: 4.5246
Epoch 2/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.0894 - loss: 4.4375
Epoch 3/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.0976 - loss: 4.3473
Epoch 4/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.1247 - loss: 4.2224
Epoch 5/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.1247 - loss: 4.1131
Epoch 6/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.1328 - loss: 3.9846
Epoch 7/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.1518 - loss: 3.8217
Epoch 8/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.1680 - loss: 3.6547
Epoch 9/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.1951 - loss: 3.4955
Epoch 10/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.2304 - loss: 3.3334
Epoch 11/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.2466 - loss: 3.1651
Epoch 12/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step

In [63]:
text = "You can do"

for i in range(5):
  #tokenization
  token_text = tokenizer.texts_to_sequences([text])[0]
  #padding
  padded_token_text = pad_sequences([token_text] , maxlen=max_len-1 , padding='pre')

  pos = np.argmax(model.predict(padded_token_text))

  for word,index in tokenizer.word_index.items():
    if index == pos:
      text = text + " " + word
      print(text)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
You can do between
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
You can do between a
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
You can do between a traditional
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
You can do between a traditional relational
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
You can do between a traditional relational database


In [56]:
import numpy as np

#How to improve Performance for this



*   Add more DaTA
*   Hyperparameter
*   Advanced Architecture

